# Open-Source Chatbot in Google Colab


It uses a **freely available Hugging Face model**, stores the model inside **Google Drive**, and loads it from Drive in **Google Colab**.

## What this notebook does
1. Mounts Google Drive
2. Downloads an open-source model to Drive only once
3. Loads the model locally from Drive
4. Builds a multi-turn chatbot
5. Optionally launches a Gradio chat interface

## Recommended models
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0` for better chat quality on Colab GPU
- `microsoft/DialoGPT-small` for a lighter fallback if GPU is unavailable

## Important
For the best experience, use **Colab with GPU**:
**Runtime -> Change runtime type -> GPU**


In [2]:
# Install required libraries in Colab
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sentencepiece gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 116.4 MB/s eta 0:00:00


## 1. Choose the model

Use one of these options:

- `tinyllama` -> better responses, recommended when GPU is available
- `dialogpt-small` -> lighter and safer fallback on weak runtimes

The model will be downloaded to Google Drive and reused in later sessions.


In [3]:
import os
import torch

MODEL_CHOICE = "tinyllama"   # change to "dialogpt-small" if needed

MODEL_MAP = {
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "dialogpt-small": "microsoft/DialoGPT-small",
}

MODEL_ID = MODEL_MAP[MODEL_CHOICE]
MODEL_NAME = MODEL_ID.split("/")[-1]
MODEL_DIR = f"/content/drive/MyDrive/colab_models/{MODEL_NAME}"

USE_GPU = torch.cuda.is_available()
print("Selected model:", MODEL_ID)
print("Model will be stored at:", MODEL_DIR)
print("GPU available:", USE_GPU)
if USE_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. You can still run the notebook, but chatbot responses may be slower.")


Selected model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Model will be stored at: /content/drive/MyDrive/colab_models/TinyLlama-1.1B-Chat-v1.0
GPU available: True
GPU: Tesla T4


## 2. Download the model to Google Drive

This downloads the model only once.  
If the files are already in Drive, the cell can be skipped.


In [4]:
from huggingface_hub import snapshot_download

os.makedirs(MODEL_DIR, exist_ok=True)

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
    resume_download=True,
    ignore_patterns=[
        "*.msgpack",
        "*.h5",
        "*.tflite",
        "*.onnx",
        "*.ot",
        "*.safetensors.index.json"  # safe to skip for these small setups
    ],
)

print("Model downloaded to:", MODEL_DIR)
print("Saved files:", os.listdir(MODEL_DIR)[:10])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model downloaded to: /content/drive/MyDrive/colab_models/TinyLlama-1.1B-Chat-v1.0
Saved files: ['.cache', 'eval_results.json', 'config.json', '.gitattributes', 'generation_config.json', 'README.md', 'tokenizer.json', 'special_tokens_map.json', 'tokenizer_config.json', 'tokenizer.model']


## 3. Load tokenizer and model from Google Drive

This section loads the model directly from the local Drive path.

- TinyLlama uses **4-bit loading** on GPU for smoother Colab usage.
- DialoGPT-small loads as a light causal language model.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)

if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

if MODEL_CHOICE == "tinyllama":
    if torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            quantization_config=quant_config,
            device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            torch_dtype=torch.float32,
        )
        model.to("cpu")
else:
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=dtype,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

model.eval()
print("Model loaded successfully.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.


## 4. Utility functions

These functions:
- build prompts
- keep chat history
- generate responses
- clean output


In [20]:
import re

SYSTEM_PROMPT = (
    "you are medical speciliest to identify the diseaase from input symptoms "
)

#  "Your are a funny character and you have tell the funny stories to kids in simple words"

# Helper function to determine the device the model is running on (GPU or CPU)
# This is useful for placing tensors on the correct device.
def _model_device():
    try:
        # Attempts to get the device of the first model parameter
        return next(model.parameters()).device
    except StopIteration:
        # If no parameters are found (e.g., model not yet loaded), defaults to CUDA if available, else CPU
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Function to construct the prompt for the chatbot model
# user_message: The current message from the user.
# history: A list of (user_message, assistant_reply) tuples representing previous turns.
#          Defaults to an empty list if not provided (history=None).
# system_prompt: An initial instruction for the chatbot's behavior.
#                Defaults to the global SYSTEM_PROMPT.
def build_prompt(user_message, history=None, system_prompt=SYSTEM_PROMPT):
    history = history or [] # Ensures history is a list, even if None is passed

    # Different prompt formats for TinyLlama and other models
    if MODEL_CHOICE == "tinyllama":
        messages = [{"role": "system", "content": system_prompt}] # Start with system prompt
        # Add historical user and assistant messages to the prompt
        for user_text, assistant_text in history:
            messages.append({"role": "user", "content": user_text})
            messages.append({"role": "assistant", "content": assistant_text})
        messages.append({"role": "user", "content": user_message}) # Add the current user message

        # Use the tokenizer's chat template for TinyLlama models
        # tokenize=False: Returns the prompt as a string, not token IDs.
        # add_generation_prompt=True: Appends a special token to indicate the model should start generating.
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return prompt # Returns the formatted prompt string

    # Fallback for models like DialoGPT-small, using a simple plain-text format
    prompt = f"System: {system_prompt}\n"
    for user_text, assistant_text in history:
        prompt += f"User: {user_text}\nAssistant: {assistant_text}\n"
    prompt += f"User: {user_message}\nAssistant:"
    return prompt # Returns the plain-text formatted prompt string

# Decorator to disable gradient calculations, saving memory and speeding up inference.
@torch.no_grad()
# Function to generate a reply from the chatbot model
# user_message: The current message from the user.
# history: A list of (user_message, assistant_reply) tuples representing previous turns.
#          Defaults to an empty list if not provided.
# system_prompt: An initial instruction for the chatbot's behavior. Defaults to SYSTEM_PROMPT.
# max_new_tokens: Maximum number of tokens the model should generate in its reply.
# temperature: Controls randomness in generations. Higher values (e.g., 1.0) make output more random.
# top_p: Filters tokens based on cumulative probability. Only tokens with a combined probability up to top_p are considered.
# repetition_penalty: Penalizes tokens that have already appeared, encouraging diversity.
def generate_reply(
    user_message,
    history=None,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
):
    history = history or [] # Ensure history is a list
    # Build the complete prompt using the helper function
    prompt = build_prompt(user_message, history=history, system_prompt=system_prompt)

    # Tokenize the prompt (convert text to numerical IDs)
    inputs = tokenizer(prompt, return_tensors="pt") # return_tensors="pt" returns PyTorch tensors
    device = _model_device() # Get the correct device (GPU/CPU)
    # Move input tensors to the model's device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate output tokens from the model
    output_ids = model.generate(
        **inputs, # Unpack input tensors into keyword arguments
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0), # Enable sampling if temperature is above 0
        temperature=max(temperature, 1e-5), # Ensure temperature is not zero for sampling
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.pad_token_id, # Specify padding token ID for generation
        eos_token_id=tokenizer.eos_token_id, # Specify end-of-sequence token ID for generation
    )

    # Extract only the newly generated tokens (excluding the input prompt tokens)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    # Decode the generated tokens back into a human-readable string
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Simple cleanup to remove potential artifacts from model generation for lighter models
    # Splits the reply at common separator patterns and takes the first part.
    reply = re.split(r"\nUser:|\nSystem:|<\|user\|>|<\|system\|>", reply)[0].strip()
    return reply # Returns the cleaned chatbot reply


## 5. Quick test


In [21]:
test_reply = generate_reply(
    "Hello. Introduce yourself in two lines.",
    history=[],
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=80,
)
print(test_reply)


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


My name is [Your Name], I am a [Your Job Title]. I work at [Company Name] as [Designation/Title]. I am responsible for [Responsibilities]. Based on the given text material, I would like to introduce myself as a Medical Specialist who specializes in identifying and treating various medical conditions using advanced diagnostic techniques, including X-


## 6. Build a multi-turn chatbot

This cell keeps conversation history across turns.
Run `chat("your message")` again and again in new cells.
Run `reset_chat()` when you want to clear memory.


In [8]:
chat_history = [] # Global list to store the conversation history

# Function to manage a multi-turn chat interaction
# user_message: The user's current input message.
# system_prompt: An optional system prompt to guide the chatbot's behavior for this turn.
#                Defaults to the global SYSTEM_PROMPT.
def chat(user_message, system_prompt=SYSTEM_PROMPT):
    global chat_history # Declare that we are using the global chat_history variable
    # Generate a reply using the generate_reply function
    reply = generate_reply(
        user_message=user_message,
        history=chat_history, # Pass the accumulated chat history
        system_prompt=system_prompt,
        max_new_tokens=160,
        temperature=0.7,
        top_p=0.9,
    )
    # Append the current user message and bot's reply to the chat history
    chat_history.append((user_message, reply))
    return reply # Return the bot's reply

# Function to clear the entire chat history
def reset_chat():
    global chat_history # Declare that we are modifying the global chat_history variable
    chat_history = [] # Reset the history to an empty list
    print("Chat history cleared.")


In [9]:
reset_chat()
print("Bot:", chat("Hi, I am Umar."))
print()
print("Bot:", chat("Can you remember my name?"))
print()
print("Bot:", chat("Explain what a transformer is in very simple words."))


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chat history cleared.


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Umar here, happy to provide you with a sample message that you can use as a starting point for your own customized chatbot. This message is an example of how you can welcome new users, provide basic information about the chatbot, and answer frequently asked questions (FAQs) in a clear, concise, and friendly manner.

Hello! Welcome to our chatbot, a platform designed to help you with all your academic needs. We understand that navigating through university life can be challenging, which is why we've created this intelligent tool to simplify your tasks. Here's a quick rundown of what you can expect from us:

1. Quickly search and find information on course-related matters such as class schedules, syllab



Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Yes, of course! Your name is "Umar". We remember everyone's names, so feel free to say hello whenever you need some help. Let's get started with your initial interactions!

Bot: Sure thing! A transformer is a kind of neural network architecture used for natural language processing (NLP). It works by learning patterns and dependencies between words, sentences, and even entire textual documents. This allows it to understand and generate context-sensitive responses based on user input. So, when you ask for help or advice related to your coursework, the chatbot will analyze the context surrounding your question and provide personalized and relevant answers. That's why we call our chatbot a "transformer" - it uses advanced techniques to extract meaning from large volumes of text data and learn from its interactions with human users. Thanks again for asking! Let's chat more soon.


## 7. Try your own chatbot persona

You can change the system prompt to specialize the bot.
Examples:
- teaching assistant
- medical imaging assistant
- coding assistant
- university helpdesk bot


In [10]:
custom_system_prompt = (
    "You are an NLP teaching assistant. "
    "Explain BERT, GPT, and Transformer concepts to beginners with short examples."
)

reset_chat()
print("Bot:", chat("What is the difference between BERT and GPT?", system_prompt=custom_system_prompt))


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chat history cleared.
Bot: BERT (Bidirectional Encoder Representations from Transformers) and GPT (Generative Pretraining) are both neural language modeling techniques that use pre-trained language models like BERT or GPT-2 to learn a large dataset for generating text. However, there are some key differences between these two methods:

1. Unlike BERT, GPT does not have pre-training on specific domains like financial, legal, or scientific texts. It uses pre-trained language models like GPT-2 to learn the language modeling task of generating text based on a given input.

2. GPT uses a sequence-to-sequence architecture, while BERT uses a transformer architecture. A transformer architecture processes the input


## 8. Save conversation history to Google Drive


In [11]:
import json

HISTORY_PATH = "/content/drive/MyDrive/colab_models/chat_history.json"

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(chat_history, f, ensure_ascii=False, indent=2)

print("Chat history saved to:", HISTORY_PATH)


Chat history saved to: /content/drive/MyDrive/colab_models/chat_history.json


## 9. Load saved conversation history


In [12]:
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        chat_history = [tuple(x) for x in json.load(f)]
    print("Loaded", len(chat_history), "turns from Drive.")
else:
    print("No saved history found yet.")


Loaded 1 turns from Drive.


## 10. Optional Gradio interface

Run this cell if you want a nicer chat UI inside Colab.

If the interface does not open inline, Colab will give you a public link.


In [15]:
import gradio as gr

def pairs_from_gradio_history(history):
    pairs = []
    current_user = None
    for item in history:
        if item["role"] == "user":
            current_user = item["content"]
        elif item["role"] == "assistant" and current_user is not None:
            pairs.append((current_user, item["content"]))
            current_user = None
    return pairs

def gradio_chat_fn(message, history):
    history = history or []
    pairs = pairs_from_gradio_history(history)
    reply = generate_reply(
        user_message=message,
        history=pairs,
        system_prompt=SYSTEM_PROMPT,
        max_new_tokens=160,
        temperature=0.7,
        top_p=0.9,
    )
    return reply

demo = gr.ChatInterface(
    fn=gradio_chat_fn,
    title="Open-Source Colab Chatbot",
    description=f"Model loaded from Google Drive: {MODEL_ID}",
)

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://948f827a92e7bfcec0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Notes

### If Colab runs out of memory
- switch `MODEL_CHOICE` to `dialogpt-small`
- reduce `max_new_tokens`
- restart runtime and load again

### If you want better quality later
You can replace the model with another open-source instruct or chat model from Hugging Face, then keep the same Google Drive download pattern.
